# Smart Warehouse Baseline (LightGBM)

결측치/이상치 처리 + GroupKFold + LightGBM으로 제출 파일을 생성합니다.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42
N_SPLITS = 3

BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / 'data'
CODE_DIR = BASE_DIR / 'code'

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
LAYOUT_PATH = DATA_DIR / 'layout_info.csv'
SUBMISSION_PATH = CODE_DIR / 'submission_test.csv'


In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
layout = pd.read_csv(LAYOUT_PATH)

print('train shape:', train.shape)
print('test shape :', test.shape)
print('layout shape:', layout.shape)


train shape: (250000, 94)
test shape : (50000, 93)
layout shape: (300, 15)


In [3]:
train = train.merge(layout, on='layout_id', how='left')
test = test.merge(layout, on='layout_id', how='left')

target_col = 'avg_delay_minutes_next_30m'
y = train[target_col].copy()
test_id = test['ID'].copy()

X_train = train.drop(columns=['ID', target_col])
X_test = test.drop(columns=['ID'])
groups = X_train['scenario_id'].copy()

cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

missing_ratio = X_train[num_cols].isna().mean()
ind_cols = missing_ratio[missing_ratio >= 0.05].index.tolist()
for c in ind_cols:
    X_train[f'is_null_{c}'] = X_train[c].isna().astype(np.int8)
    X_test[f'is_null_{c}'] = X_test[c].isna().astype(np.int8)

num_cols = [c for c in X_train.columns if c not in cat_cols]
medians = X_train[num_cols].median(numeric_only=True)
X_train[num_cols] = X_train[num_cols].fillna(medians)
X_test[num_cols] = X_test[num_cols].fillna(medians)

for c in cat_cols:
    X_train[c] = X_train[c].fillna('missing').astype(str)
    X_test[c] = X_test[c].fillna('missing').astype(str)

for c in cat_cols:
    freq = X_train[c].value_counts(normalize=True)
    X_train[c] = X_train[c].map(freq).fillna(0).astype(np.float32)
    X_test[c] = X_test[c].map(freq).fillna(0).astype(np.float32)

all_num_cols = X_train.columns.tolist()
lower = X_train[all_num_cols].quantile(0.001)
upper = X_train[all_num_cols].quantile(0.999)
X_train[all_num_cols] = X_train[all_num_cols].clip(lower=lower, upper=upper, axis=1)
X_test[all_num_cols] = X_test[all_num_cols].clip(lower=lower, upper=upper, axis=1)

print('final train shape:', X_train.shape)
print('final test shape :', X_test.shape)


final train shape: (250000, 192)
final test shape : (50000, 192)


In [4]:
gkf = GroupKFold(n_splits=N_SPLITS)
oof = np.zeros(len(X_train), dtype=np.float64)
pred_test = np.zeros(len(X_test), dtype=np.float64)

params = dict(
    objective='mae',
    metric='l1',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=80,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y, groups), 1):
    print(f'--- Fold {fold} ---')
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric='l1',
        callbacks=[early_stopping(50), log_evaluation(100)],
    )

    oof[va_idx] = model.predict(X_va, num_iteration=model.best_iteration_)
    pred_test += model.predict(X_test, num_iteration=model.best_iteration_) / N_SPLITS

oof_mae = mean_absolute_error(y, oof)
print(f'OOF MAE: {oof_mae:.5f}')


--- Fold 1 ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036813 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 21325
[LightGBM] [Info] Number of data points in the train set: 166650, number of used features: 191
[LightGBM] [Info] Start training from score 8.952078
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 9.61208
[200]	valid_0's l1: 9.59561
Early stopping, best iteration is:
[225]	valid_0's l1: 9.59129
--- Fold 2 ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.033601 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 21330
[LightGBM] [Info] Number of data points in the train set: 166675, number of used features: 191
[LightGBM] [Info] Start t

In [5]:
submission = pd.DataFrame({
    'ID': test_id,
    'avg_delay_minutes_next_30m': pred_test
})
submission.to_csv(SUBMISSION_PATH, index=False)
print('saved:', SUBMISSION_PATH)
submission.head()


saved: ../code/submission_test.csv


,ID,avg_delay_minutes_next_30m
0,TEST_000000,15.871508
1,TEST_000001,15.160585
2,TEST_000002,17.501996
3,TEST_000003,16.875034
4,TEST_000004,16.328431
